In [1]:
%load_ext autoreload
%autoreload 2

In [29]:
import sys 
sys.path.append('../src')

import os
import pathlib
import tqdm as tqdm
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

import torch
from rescue.lang_features import LSegLangFeatures
from sklearn.decomposition import PCA, IncrementalPCA
from rescue.feature_reduction import TorchIncrementalPCA

In [31]:
lang_feats = LSegLangFeatures(checkpoint_path="../generated/lseg_minimal_e200.ckpt",)

In [ ]:
num_imgs = -1
ipca = TorchIncrementalPCA(n_components=128, device='cuda')

img_paths = list(pathlib.Path("../generated/sampled_disaster_city").glob("*.jpg"))[:num_imgs]

for img_path in tqdm.tqdm(img_paths):
    img_feat = lang_feats.extract_dense_features(img_path)
    img_feat_flat = torch.permute(img_feat, (0, 2, 3, 1)).reshape(-1, img_feat.shape[1])    # (76800, 512)
    ipca.partial_fit(img_feat_flat)  

100%|██████████| 204/204 [00:43<00:00,  4.74it/s]


In [ ]:
for img_path in tqdm.tqdm(img_paths):
    img_feat = lang_feats.extract_dense_features(img_path)
    img_feat_flat = torch.permute(img_feat, (0, 2, 3, 1)).reshape(-1, img_feat.shape[1])    # (76800, 512)
    
    ipca.partial_fit(img_feat_flat.cpu().numpy())  

In [ ]:
# ipca = IncrementalPCA(n_components=128)  # no batch_size needed

# img_paths = list(pathlib.Path("../generated/sampled_disaster_city").glob("*.jpg"))[:10]



  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:32<00:00,  3.20s/it]


In [ ]:
# print(f"cumulative variance = {ipca.explained_variance_ratio_.cumsum()[-1]:.3f}")

cumulative variance = 1.000


In [11]:
# for img_path in img_paths:
#     img_feat = lang_feats.extract_dense_features(img_path)
#     img_feat_flat = torch.permute(img_feat, (0, 2, 3, 1)).reshape(-1, img_feat.shape[1])    
#     reduced = ipca.transform(img_feat_flat.cpu().numpy()) 
#     print(reduced.shape)

In [ ]:
# import numpy as np

# float_size = np.dtype(np.float32).itemsize  # 4 bytes

# shape1 = (76800, 512)  # original features
# shape2 = (76800, 128)   # reduced features

# size1_bytes = np.prod(shape1) * float_size
# size2_bytes = np.prod(shape2) * float_size

# size1_mb = size1_bytes / (1024 ** 2)
# size2_mb = size2_bytes / (1024 ** 2)

# print(f"Original shape {shape1}: {size1_bytes} bytes ({size1_mb:.2f} MB)")
# print(f"Reduced shape {shape2}: {size2_bytes} bytes ({size2_mb:.2f} MB)")
# print(f"Reduction factor: {size1_bytes/size2_bytes:.1f}x")

Original shape (76800, 512): 157286400 bytes (150.00 MB)
Reduced shape (76800, 128): 39321600 bytes (37.50 MB)
Reduction factor: 4.0x


cumulative variance = 0.980
